In [ ]:
import requests
import time
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from shapely.geometry import shape

# Importation des pays

<span style="font-size:1.5em;">
Rest Countries
</span>

In [164]:
url__pays_restcountries = "https://restcountries.com/v3.1/region/europe?fields=name,cca2,cca3,translations"

response = requests.get(url__pays_restcountries)

if response.status_code == 200:
    countries = response.json()
    
    pays_restcountries = {
        country['cca2']: country['name']['common']
        for country in countries
        if 'cca2' in country and 'name' in country
    }

    pays_restcountries_cca2_3 = {
        country['cca2']: country['cca3']
        for country in countries
        if 'cca2' in country and 'cca3' in country
    }

    pays_restcountries_fr = {
        country['cca2']: country['translations']['fra']['common']
        for country in countries
        if 'cca2' in country and 'translations' in country and 'fra' in country['translations']
    }
    
    print(pays_restcountries)
else:
    print(f"Erreur: {response.status_code}")

{'ES': 'Spain', 'NL': 'Netherlands', 'LT': 'Lithuania', 'IT': 'Italy', 'CH': 'Switzerland', 'SJ': 'Svalbard and Jan Mayen', 'FI': 'Finland', 'DE': 'Germany', 'LV': 'Latvia', 'IM': 'Isle of Man', 'MT': 'Malta', 'GB': 'United Kingdom', 'GR': 'Greece', 'CZ': 'Czechia', 'JE': 'Jersey', 'AL': 'Albania', 'PT': 'Portugal', 'SI': 'Slovenia', 'SM': 'San Marino', 'BE': 'Belgium', 'FR': 'France', 'BG': 'Bulgaria', 'CY': 'Cyprus', 'VA': 'Vatican City', 'MK': 'North Macedonia', 'XK': 'Kosovo', 'GI': 'Gibraltar', 'DK': 'Denmark', 'ME': 'Montenegro', 'AT': 'Austria', 'SK': 'Slovakia', 'AX': 'Åland Islands', 'AD': 'Andorra', 'MD': 'Moldova', 'RU': 'Russia', 'LI': 'Liechtenstein', 'EE': 'Estonia', 'RO': 'Romania', 'BY': 'Belarus', 'FO': 'Faroe Islands', 'UA': 'Ukraine', 'MC': 'Monaco', 'HU': 'Hungary', 'BA': 'Bosnia and Herzegovina', 'LU': 'Luxembourg', 'RS': 'Serbia', 'PL': 'Poland', 'SE': 'Sweden', 'IE': 'Ireland', 'IS': 'Iceland', 'NO': 'Norway', 'GG': 'Guernsey', 'HR': 'Croatia'}


<span style="font-size:1.5em;">
Data Country
</span>

In [3]:
url__pays_datacountry = "https://api.datacountry.io/v1/countries/region/Europe"

response = requests.get(url__pays_datacountry)

if response.status_code == 200:
    result = response.json()
    
    countries = result.get("data", [])
    
    pays_datacountry = {
        country['cca2']: country['name']['common']
        for country in countries
        if country.get("cca2") and country.get("name")
    }

    print(pays_datacountry)

else:
    print(f"Erreur: {response.status_code}")

{'AL': 'Albania', 'AD': 'Andorra', 'AT': 'Austria', 'BY': 'Belarus', 'BE': 'Belgium', 'BA': 'Bosnia and Herzegovina', 'BG': 'Bulgaria', 'HR': 'Croatia', 'CY': 'Cyprus', 'CZ': 'Czechia', 'DK': 'Denmark', 'EE': 'Estonia', 'FO': 'Faroe Islands', 'FI': 'Finland', 'FR': 'France', 'DE': 'Germany', 'GI': 'Gibraltar', 'GR': 'Greece', 'GG': 'Guernsey', 'HU': 'Hungary', 'IS': 'Iceland', 'IE': 'Ireland', 'IM': 'Isle of Man', 'IT': 'Italy', 'JE': 'Jersey', 'XK': 'Kosovo', 'LV': 'Latvia', 'LI': 'Liechtenstein', 'LT': 'Lithuania', 'LU': 'Luxembourg', 'MT': 'Malta', 'MD': 'Moldova', 'MC': 'Monaco', 'ME': 'Montenegro', 'NL': 'Netherlands', 'MK': 'North Macedonia', 'NO': 'Norway', 'PL': 'Poland', 'PT': 'Portugal', 'RO': 'Romania', 'RU': 'Russia', 'SM': 'San Marino', 'RS': 'Serbia', 'SK': 'Slovakia', 'SI': 'Slovenia', 'ES': 'Spain', 'SJ': 'Svalbard and Jan Mayen', 'SE': 'Sweden', 'CH': 'Switzerland', 'UA': 'Ukraine', 'GB': 'United Kingdom', 'VA': 'Vatican City', 'AX': 'Åland Islands'}


On pourrait chercher le nom du pays en francais avec : [https://api.datacountry.io/v1/countries/`{cca2}`?include=translations](), mais _datacountry_ limite le nombre de requête à 50/jour et l'Europe a plus de 50 pays...

<span style="font-size:1.5em;"> 
Vérification
</span>

On regarde si les 2 API sont d'accord sur les pays Européen.

In [4]:
pays_restcountries == pays_datacountry

True

In [45]:
len(pays_datacountry)

53

# HDI : Indice de Développement Humain

## Formules

L’IDH est défini comme la moyenne géométrique de trois indices normalisés : voir [HDI : Human development Reports](https://hdr.undp.org/data-center/human-development-index#/indicies/HDI).


<span style="font-size:1.5em;">
    Formule générale
</span>


$$
\boxed{
HDI = \sqrt[3]{ I_{\text{santé}} \times I_{\text{éducation}} \times I_{\text{revenu}} }
}
$$

- Indice de santé :

    $
    \boxed{
    I_{\text{santé}} = \frac{LE - 20}{85 - 20}
    }
    $
    - **LE** = espérance de vie à la naissance
    - Bornes : 20 → 85 ans

- Indice d’éducation :

    $
    \boxed{
    I_{\text{éducation}} = \frac{ I_{\text{scolarité moyenne}} + I_{\text{scolarité attendue}} }{2}
    }
    $

    - $ \boxed{ I_{\text{scolarité moyenne}} = \frac{MS}{15} } $
    - $ \boxed{ I_{\text{scolarité attendue}} = \frac{ES}{18} } $

        - MS = années moyennes de scolarisation
        - ES = années de scolarisation attendues

- Indice de revenu

    $
    \boxed{
    I_{\text{revenu}} = \frac{\ln(GNI_{\text{pc}}) - \ln(100)}{\ln(75000) - \ln(100)}
    }
    $

    * $GNI_{\text{pc}}$ = revenu national brut par habitant
    * Logarithme utilisé pour réduire l’effet des très hauts revenus



<span style="font-size:1.5em;">
    Approxymation
</span>


Avec World Bank dataset :

- pas MS / ES → utilisation d'un proxy d’éducation
- remplacement de GNI par RNB

## Calculs

In [408]:
def get_indicator(cca2, indicateur, compte = 1):
    url_hdi_worldbank = f"https://api.worldbank.org/v2/country/{cca2}/indicator/{indicateur}?format=json&per_page=20000"

    try:
        response = requests.get(url_hdi_worldbank)
        
        if response.status_code != 200:
            raise Exception(f"HTTP {response.status_code}")
        
        data = response.json()
        
        if len(data) < 2:
            return []
        
        return data[1]
    
    except Exception as e:
        print(f"Tentative {compte} échouée pour {cca2} : {e}")
        
        if compte <= 3:
            time.sleep(5)
            return get_indicator(cca2, indicateur, compte + 1)
        
        return []

In [ ]:
indicators = {
    "PIB_par_habitant": "NY.GDP.PCAP.CD",
    "PIB_total": "NY.GDP.MKTP.CD",
    "RNB_par_habitant": "NY.GNP.PCAP.CD",
    "espérance_de_vie": "SP.DYN.LE00.IN",
    "mortalité_infantile": "SP.DYN.IMRT.IN",
    "dépenses_de_santé": "SH.XPD.CHEX.GD.ZS",
    "médecins_pour_1000": "SH.MED.PHYS.ZS",
    "enseignement_primaire": "SE.PRM.ENRR",
    "enseignement_secondaire": "SE.SEC.ENRR",
    "dépenses_education": "SE.XPD.TOTL.GD.ZS"
}

rows = {}

for cca2, pays in pays_datacountry.items() : 
    for ind_name, ind in indicators.items():
        data = get_indicator(cca2, ind)
        
        if not data:
            continue
        
        for obs in data:
            if obs["value"] is None:
                continue

            year = int(obs["date"])

            key = (pays, cca2, year)

            if key not in rows:
                rows[key] = {
                    "country": pays,
                    "cca2": cca2,
                    "year": year
                }

            rows[key][ind_name] = obs["value"]


df1 = pd.DataFrame(rows.values())


In [ ]:
df_hdi = df1.copy()

df_hdi["I_sante"] = (df_hdi["espérance_de_vie"] - 20) / (85 - 20)
#df_hdi["I_sante"] = np.clip(df_hdi["I_sante"], 0, 1)

df_hdi["I_revenu"] = (np.log(df_hdi["RNB_par_habitant"]) - np.log(100)) / (np.log(75000) - np.log(100))
#df_hdi["I_revenu"] = np.clip(df_hdi["I_revenu"], 0, 1)

df_hdi["I_education"] = ( 
    df_hdi["enseignement_primaire"] / 100 +
    df_hdi["enseignement_secondaire"] / 100+
    df_hdi["dépenses_education"] / 100
) / 3
# Divise par 100 car donnée en % et on veut un indice entre 0 et 1

In [165]:
df_hdi["HDI"] = (df_hdi["I_sante"] * df_hdi["I_education"] * df_hdi["I_revenu"])**(1/3)

In [166]:
df_hdi['country_fr'] = df_hdi['cca2'].map(pays_restcountries_fr)
df_hdi['cca3'] = df_hdi['cca2'].map(pays_restcountries_cca2_3)

In [167]:
nan_par_annee = df_hdi.groupby('year')['HDI'].apply(lambda x: x.isna().sum())
annees_valides = nan_par_annee[nan_par_annee < 25].index

df_hdi = df_hdi[df_hdi['year'].isin(annees_valides)].copy()

In [168]:
df_hdi = df_hdi[["country", "country_fr", "cca2", "cca3", "year", "HDI"]].copy()
print(f"Début des données {df_hdi["year"].min()}, et dernière données {df_hdi["year"].max()}")

Début des données 2001, et dernière données 2022


In [171]:
# Top 10 des valeurs pour l'année 2022
df_hdi_sorted = df_hdi[df_hdi['year'] == 2022].sort_values(by="HDI", ascending=False)
print(df_hdi_sorted.head(10))

          country  country_fr cca2 cca3  year       HDI
2862       Sweden       Suède   SE  SWE  2022  0.938394
2082  Netherlands    Pays-Bas   NL  NLD  2022  0.913295
262       Belgium    Belgique   BE  BEL  2022  0.911469
847       Finland    Finlande   FI  FIN  2022  0.906675
652       Denmark    Danemark   DK  DNK  2022  0.904639
2212       Norway     Norvège   NO  NOR  2022  0.895264
1237      Iceland     Islande   IS  ISL  2022  0.885324
2927  Switzerland      Suisse   CH  CHE  2022  0.880973
1757   Luxembourg  Luxembourg   LU  LUX  2022  0.879940
2797        Spain     Espagne   ES  ESP  2022  0.863283


In [172]:
# Bottom 10 des valeurs pour l'année 2022
df_hdi_sorted = df_hdi[df_hdi['year'] == 2022].sort_values(by="HDI", ascending=True)
print(df_hdi_sorted.head(10))

                     country          country_fr cca2 cca3  year       HDI
1887                 Moldova            Moldavie   MD  MDA  2022  0.696404
197                  Belarus         Biélorussie   BY  BLR  2022  0.709407
327   Bosnia and Herzegovina  Bosnie-Herzégovine   BA  BIH  2022  0.715428
2407                 Romania            Roumanie   RO  ROU  2022  0.716845
2602                  Serbia              Serbie   RS  SRB  2022  0.721722
2                    Albania             Albanie   AL  ALB  2022  0.729779
2472                  Russia              Russie   RU  RUS  2022  0.730323
392                 Bulgaria            Bulgarie   BG  BGR  2022  0.738688
1562                  Latvia            Lettonie   LV  LVA  2022  0.775955
2667                Slovakia           Slovaquie   SK  SVK  2022  0.776888


In [173]:
# Top 5 des valeurs moyennes par pays
df_hdi_mean = df_hdi.groupby('country')['HDI'].mean().reset_index()

# Top 5 des HDI moyens
top5 = df_hdi_mean.nlargest(5, 'HDI')
print("Top 5 des pays par HDI moyen :")
print(top5)

# Bottom 5 des HDI moyens
bottom5 = df_hdi_mean.nsmallest(5, 'HDI')
print("\nBottom 5 des pays par HDI moyen :")
print(bottom5)

Top 5 des pays par HDI moyen :
          country       HDI
4         Belgium  0.916383
44         Sweden  0.892392
34         Norway  0.881107
25  Liechtenstein  0.880899
10        Denmark  0.879996

Bottom 5 des pays par HDI moyen :
            country       HDI
33  North Macedonia  0.607361
46          Ukraine  0.620113
29          Moldova  0.657883
0           Albania  0.684553
37          Romania  0.689925


## Affichage

<span style="font-size:1.5em;">
    Cartes
</span>

In [ ]:
geo_url_long = "https://overpass.openstreetmap.ru/api/interpreter"

query = """
[out:json][timeout:300];
relation["admin_level"="2"]["ISO3166-1:alpha2"="FR"];
out geom;
"""

response = requests.post(geo_url_long, data=query, headers={
    "User-Agent": "Mozilla/5.0"
})

print("status:", response.status_code)

if "application/json" not in response.headers.get("Content-Type", ""):
    print(response.text[:500])
    raise Exception("Overpass n'a pas renvoyé du JSON")

data_geo_url_long = response.json()

ConnectTimeout: HTTPSConnectionPool(host='overpass.openstreetmap.ru', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x1210bbed0>, 'Connection to overpass.openstreetmap.ru timed out. (connect timeout=None)'))

In [232]:
geo_url = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json"
geojson = requests.get(geo_url).json()

rows = []
for feature in geojson['features']:
    row = {
        'type': feature['type'],
        'id': feature['id'],
        'name': feature['properties']['name'],
        'geometry': feature['geometry']
    }
    rows.append(row)

# Créer le DataFrame
gdf = pd.DataFrame(rows)
print(gdf.head())

      type   id                  name  \
0  Feature  AFG           Afghanistan   
1  Feature  AGO                Angola   
2  Feature  ALB               Albania   
3  Feature  ARE  United Arab Emirates   
4  Feature  ARG             Argentina   

                                            geometry  
0  {'type': 'Polygon', 'coordinates': [[[61.21081...  
1  {'type': 'MultiPolygon', 'coordinates': [[[[16...  
2  {'type': 'Polygon', 'coordinates': [[[20.59024...  
3  {'type': 'Polygon', 'coordinates': [[[51.57951...  
4  {'type': 'MultiPolygon', 'coordinates': [[[[-6...  


In [233]:
gdf['geometry'] = gdf['geometry'].apply(shape)
gdf = gpd.GeoDataFrame(gdf, geometry='geometry')

In [234]:
# On garde nos pays d'Europe
pays_iso3 = pays_restcountries_cca2_3.values()
pays_noms = pays_datacountry.values()

# Ajuster les données gdf
for idx, row in gdf.iterrows():
    nom = row['name']
    iso3 = row['id']

    # Si on a le nom correct mais pas le code
    if nom in pays_noms and iso3 not in pays_iso3:
        # Trouver le code correspondant au nom du pays
        iso2 = next((k for k, v in pays_datacountry.items() if v == nom), None)
        iso3 = pays_restcountries_cca2_3.get(iso2) if iso2 else None
        if iso3:
            gdf.at[idx, 'id'] = iso3

    # Si le code est correct mais pas le nom
    if iso3 in pays_iso3 and nom not in pays_noms:
        # Trouver le nom correspondant au code du pays
        iso2 = next((k for k, v in pays_restcountries_cca2_3.items() if v == iso3), None)
        nom_pays = pays_datacountry.get(iso2) if iso2 else None
        if nom_pays:
            gdf.at[idx, 'name'] = nom_pays

# Filtrer les pays d'Europe
europe = gdf[gdf['id'].isin(pays_iso3) | gdf['name'].isin(pays_noms)]
print(f"Perte de {len(pays_iso3) - len(europe)} pays.")
iso_filtre = europe['id'].values
noms_filtre = europe['name'].values

iso_set = set(iso_filtre)
noms_set = set(noms_filtre)

pays_non_filtrés = {iso: nom for iso, nom in pays_datacountry.items()
                    if iso not in iso_set and nom not in noms_set}

print("Liste des pays absents :")
i = 0
for key, value in pays_non_filtrés.items():
    i += 1
    print(f"{i}. {key} : {value}")

Perte de 12 pays.
Liste des pays absents :
1. AD : Andorra
2. FO : Faroe Islands
3. GI : Gibraltar
4. GG : Guernsey
5. IM : Isle of Man
6. JE : Jersey
7. LI : Liechtenstein
8. MC : Monaco
9. SM : San Marino
10. SJ : Svalbard and Jan Mayen
11. VA : Vatican City
12. AX : Åland Islands


<span style="font-size:1.2em;">
    HDI moyens
</span>

In [315]:
# Calculer la moyenne HDI
hdi_mean = df_hdi.groupby('cca3').agg({
    'HDI': 'mean',
    'country_fr': 'first' 
}).reset_index()

# Fusionner les données HDI avec le GeoDataFrame
europe_hdi = europe.merge(hdi_mean, left_on='id', right_on='cca3', how='left')

# Liste des pays avec HDI
with_hdi = europe_hdi[~europe_hdi['HDI'].isna()]
without_hdi = europe_hdi[europe_hdi['HDI'].isna()]

# Centre de la carte
center_lon = (45 + -25) / 2
center_lat = (72 + 34) / 2

# Création de la figure Plotly
fig = go.Figure()

# Couche HDI
fig.add_trace(go.Choropleth(
    geojson=with_hdi.__geo_interface__,
    locations=with_hdi.index,
    z=with_hdi['HDI'],
    text=with_hdi['country_fr'],
    colorscale='YlGn',
    colorbar_title="HDI moyen",
    marker_line_color='white',
    marker_line_width=0.5,
    hovertemplate="<b>%{text}</b><br>HDI: %{z}<extra></extra>",  # Info bulle pour HDI connu
))

# Couche gris pour pays sans HDI
fig.add_trace(go.Choropleth(
    geojson=without_hdi.__geo_interface__,
    locations=without_hdi.index,
    z=[0]*len(without_hdi),  # valeur arbitraire
    text=without_hdi['country_fr'],
    colorscale=[[0, 'lightgrey'], [1, 'lightgrey']],  # gris fixe
    showscale=False,
    marker_line_color='white',
    marker_line_width=0.5,
    hovertemplate="<b>%{text}</b><br>HDI: Null<extra></extra>",  # Info bulle pour HDI manquant
))

fig.update_geos(
    visible=False,
    projection_scale=5,
    center={"lat": center_lat, "lon": center_lon},
)

fig.update_layout(
    title=f"Carte de l'Europe colorée par HDI moyen de {df_hdi['year'].min()} à {df_hdi['year'].max()}",
    margin={"r":0, "t":50, "l":0, "b":0},
)

fig.show()

<span style="font-size:1.5em;">
    LineCharts
</span>

In [238]:
# Palette de 10 couleurs
colors = px.colors.qualitative.Plotly  

df_top5 = df_hdi[df_hdi['country'].isin(top5['country'])]
df_bottom5 = df_hdi[df_hdi['country'].isin(bottom5['country'])]

# Combiner les deux DataFrames pour avoir toutes les lignes
df_combined = pd.concat([df_top5, df_bottom5])
fig = go.Figure()

# Parcourir chaque pays
for i, (country, group) in enumerate(df_combined.groupby('country')):
    group = group.sort_values('year')
    years = group['year'].values
    hdi = group['HDI'].values
    country_fr = group['country_fr'].iloc[0]

    mask_valid = ~np.isnan(hdi)

    # Tracer la ligne continue pour les valeurs existantes
    fig.add_trace(go.Scatter(
        x=years[mask_valid],
        y=hdi[mask_valid],
        mode='lines+markers',
        name=country_fr,
        line=dict(color=colors[i % len(colors)]),  # couleur distincte
        marker=dict(color=colors[i % len(colors)])
    ))

    # Tracer les segments pointillés là où il y a des valeurs manquantes
    for j in range(1, len(years)):
        if mask_valid[j] and not mask_valid[j-1]:
            prev_idx = j - 1
            while prev_idx >= 0 and not mask_valid[prev_idx]:
                prev_idx -= 1
            if prev_idx >= 0:
                fig.add_trace(go.Scatter(
                    x=[years[prev_idx], years[j]],
                    y=[hdi[prev_idx], hdi[j]],
                    mode='lines',
                    line=dict(dash='dash', color="white"),  # segment pointillé avec même couleur
                    showlegend=False
                ))

# Ajouter la ligne moyenne Europe
df_europe_mean = df_hdi.groupby('year')['HDI'].mean().reset_index()
fig.add_trace(go.Scatter(
    x=df_europe_mean['year'],
    y=df_europe_mean['HDI'],
    mode='lines+markers',
    name='Moyenne Europe',
    line=dict(color='black', width=3, dash='dot'),
))

# Mise en page
fig.update_layout(
    title="Évolution du HDI - Les 5 meilleurs et pires",
    xaxis_title="Année",
    yaxis_title="HDI",
    legend_title="Pays",
    template="plotly_white",
    width=900,
    height=500
)

fig.show()

# BLI : Better Life Index

## Formule

Le BLI est défini a somme pondérée de plusieurs dimensions du bien-être, chacune étant normalisée : voir [BLI : World Happiness Report](https://www.worldhappiness.report).

<span style="font-size:1.5em;">
    Formule générale
</span>

$$
\boxed{
BLI = \sum_{k=1}^{11} w_k \cdot D_k
}
$$

* $D_k$ = dimension du bien-être
* $w_k$ = poids associé à la dimension $k$
* $\sum_{k=1}^{11} w_k = 1$

<span style="font-size:1.5em;">
    Les dimensions
</span>

- $D_1$ : Revenu
- $D_2$ : Emploi
- $D_3$ : Logement
- $D_4$ : Santé
- $D_5$ : Éducation
- $D_6$ : Environnement
- $D_7$ : Engagement civique
- $D_8$ : Communauté
- $D_9$ : Sécurité
- $D_{10}$ : Satisfaction de vie
- $D_{11}$ : Équilibre vie professionnelle / vie personnelle

<span style="font-size:1.5em;">
    Approxymation
</span>


Avec les API :

- Sécurité ($D_9$) → nombre de policiers (forces de sécurité) et dépenses en sécurité publique


Perte des dimensions :

- Engagement civique ($D_7$)
- Communauté ($D_8$)
- Satisfaction de vie ($D_{10}$)
- Équilibre vie professionnelle / vie personnelle ($D_{11}$)

## Calculs

In [244]:
indicateurs = {
    "revenu": "NY.GDP.PCAP.CD",  # D1

    "emploi": "SL.TLF.ACTI.ZS",  # D2

    "acces_electricite": "EG.ELC.ACCS.ZS",  # D3
    "acces_eau": "SH.H2O.BASW.ZS",  # D3

    "depenses_sante": "SH.XPD.CHEX.PC.CD",  # D4
    "esperance_vie": "SP.DYN.LE00.IN",  # D4
    "mortalite_infantile": "SP.DYN.IMRT.IN",  # D4
    "medecins": "SH.MED.PHYS.ZS",  # D4

    "primaire": "SE.PRM.ENRR",  # D5
    "secondaire": "SE.SEC.ENRR",  # D5
    "depenses_education": "SE.XPD.TOTL.GD.ZS",  # D5

    "pollution_air": "EN.ATM.PM25.MC.M3",  # D6

    "homicides": "VC.IHR.PSRC.P5"  # D9
}

rows = {}

for cca2, pays in pays_datacountry.items() : 
    for ind_name, ind in indicateurs.items():
        data = get_indicator(cca2, ind)
        
        if not data:
            continue
        
        for obs in data:
            if obs["value"] is None:
                continue

            year = int(obs["date"])

            key = (pays, cca2, year)

            if key not in rows:
                rows[key] = {
                    "country": pays,
                    "cca2": cca2,
                    "year": year
                }

            rows[key][ind_name] = obs["value"]


df2 = pd.DataFrame(rows.values())

Tentative 1 échouée pour CH : HTTP 502
Tentative 1 échouée pour UA : HTTP 504
Tentative 1 échouée pour UA : HTTP 500


In [462]:
df_bli = df2.copy()

In [463]:
# colonnes à NE PAS normaliser
exclude = ["year"]

# garder seulement les colonnes numériques sauf "year"
num_cols = df_bli.select_dtypes(include="number").columns
cols_to_norm = [c for c in num_cols if c not in exclude]

# min-max scaling
df_bli[cols_to_norm] = (
    df_bli[cols_to_norm] - df_bli[cols_to_norm].min()
) / (df_bli[cols_to_norm].max() - df_bli[cols_to_norm].min())


df_bli["mortalite_infantile"] = 1 - df_bli["mortalite_infantile"]
df_bli["pollution_air"] = 1 - df_bli["pollution_air"]
df_bli["homicides"] = 1 - df_bli["homicides"]

In [464]:
df_bli["logement"] = (
    df_bli["acces_electricite"] + 
    df_bli["acces_eau"]
) / 2

df_bli["sante"] = (
    df_bli["depenses_sante"] +
    df_bli["esperance_vie"] + 
    df_bli["mortalite_infantile"] + 
    df_bli["medecins"]
) / 4

df_bli["education"] = (
    df_bli["primaire"]+ 
    df_bli["secondaire"] + 
    df_bli["depenses_education"]
) / 3

df_bli["environnement"] = (
    df_bli["pollution_air"] 
) / 1

df_bli = df_bli.rename(columns={
    "homicides": "securite"
})

In [465]:
df_bli["BLI"] = (
    df_bli["revenu"]
    + df_bli["emploi"]
    + df_bli["logement"]
    + df_bli["sante"]
    + df_bli["education"]
    + df_bli["environnement"]
    + df_bli["securite"]
)

In [466]:
df_bli['country_fr'] = df_bli['cca2'].map(pays_restcountries_fr)
df_bli['cca3'] = df_bli['cca2'].map(pays_restcountries_cca2_3)

In [467]:
nan_par_annee = df_bli.groupby('year')['BLI'].apply(lambda x: x.isna().sum())
annees_valides = nan_par_annee[nan_par_annee < 25].index

df_bli = df_bli[df_bli['year'].isin(annees_valides)].copy()

In [468]:
df_bli = df_bli[["country", "country_fr", "cca2", "cca3", "year", "BLI"]].copy()
print(f"Début des données {df_bli["year"].min()}, et dernière données {df_bli["year"].max()}")

Début des données 2001, et dernière données 2020


In [469]:
# Top 10 des valeurs pour la dernière année
df_bli_sorted = df_bli[df_bli['year'] == df_bli["year"].max()].sort_values(by="BLI", ascending=False)
print(df_bli_sorted.head(10))

             country   country_fr cca2 cca3  year       BLI
2900          Sweden        Suède   SE  SWE  2020  5.381782
1255         Iceland      Islande   IS  ISL  2020  5.364904
2966     Switzerland       Suisse   CH  CHE  2020  5.320853
2241          Norway      Norvège   NO  NOR  2020  5.303560
663          Denmark     Danemark   DK  DNK  2020  5.124028
2109     Netherlands     Pays-Bas   NL  NLD  2020  5.102098
860          Finland     Finlande   FI  FIN  2020  5.092180
992          Germany    Allemagne   DE  DEU  2020  4.887151
135          Austria     Autriche   AT  AUT  2020  4.842143
3097  United Kingdom  Royaume-Uni   GB  GBR  2020  4.811796


In [470]:
# Bottom 10 des valeurs pour la première année
df_bli_sorted = df_bli[df_bli['year'] == df_bli["year"].max()].sort_values(by="BLI", ascending=True)
print(df_bli_sorted.head(10))

       country country_fr cca2 cca3  year       BLI
4      Albania    Albanie   AL  ALB  2020  3.836880
2439   Romania   Roumanie   RO  ROU  2020  3.889078
2307    Poland    Pologne   PL  POL  2020  3.901569
1912   Moldova   Moldavie   MD  MDA  2020  3.935920
2505    Russia     Russie   RU  RUS  2020  4.080666
465    Croatia    Croatie   HR  HRV  2020  4.105390
399   Bulgaria   Bulgarie   BG  BGR  2020  4.143222
1452     Italy     Italie   IT  ITA  2020  4.214947
2702  Slovakia  Slovaquie   SK  SVK  2020  4.261364
1189   Hungary    Hongrie   HU  HUN  2020  4.334170


In [471]:
# Top 5 des valeurs moyennes par pays
df_bli_mean = df_bli.groupby('country')['BLI'].mean().reset_index()

# Top 5 des BLI moyens
top5 = df_bli_mean.nlargest(5, 'BLI')
print("Top 5 des pays par BLI moyen :")
print(top5)

# Bottom 5 des BLI moyens
bottom5 = df_bli_mean.nsmallest(5, 'BLI')
print("\nBottom 5 des pays par BLI moyen :")
print(bottom5)

Top 5 des pays par BLI moyen :
        country       BLI
19      Iceland  5.298316
34       Norway  5.214152
44       Sweden  5.159304
45  Switzerland  5.093903
10      Denmark  4.978087

Bottom 5 des pays par BLI moyen :
                   country       BLI
33         North Macedonia  2.898771
5   Bosnia and Herzegovina  3.213231
40                  Serbia  3.299173
0                  Albania  3.444213
37                 Romania  3.518408


## Affichage

In [472]:
# Calculer la moyenne BLI
bli_mean = df_bli.groupby('cca3').agg({
    'BLI': 'mean',
    'country_fr': 'first' 
}).reset_index()

# Fusionner les données BLI avec le GeoDataFrame
europe_bli = europe.merge(bli_mean, left_on='id', right_on='cca3', how='left')

# Liste des pays avec BLI
with_bli = europe_bli[~europe_bli['BLI'].isna()]
without_bli = europe_bli[europe_bli['BLI'].isna()]

# Centre de la carte
center_lon = (45 + -25) / 2
center_lat = (72 + 34) / 2

# Création de la figure Plotly
fig = go.Figure()

# Couche BLI
fig.add_trace(go.Choropleth(
    geojson=with_bli.__geo_interface__,
    locations=with_bli.index,
    z=with_bli['BLI'],
    text=with_bli['country_fr'],
    colorscale='YlGn',
    colorbar_title="BLI moyen",
    marker_line_color='white',
    marker_line_width=0.5,
    hovertemplate="<b>%{text}</b><br>BLI: %{z}<extra></extra>",  # Info bulle pour BLI connu
))

# Couche gris pour pays sans BLI
fig.add_trace(go.Choropleth(
    geojson=without_bli.__geo_interface__,
    locations=without_bli.index,
    z=[0]*len(without_bli),  # valeur arbitraire
    text=without_bli['country_fr'],
    colorscale=[[0, 'lightgrey'], [1, 'lightgrey']],  # gris fixe
    showscale=False,
    marker_line_color='white',
    marker_line_width=0.5,
    hovertemplate="<b>%{text}</b><br>BLI: Null<extra></extra>",  # Info bulle pour BLI manquant
))

fig.update_geos(
    visible=False,
    projection_scale=5,
    center={"lat": center_lat, "lon": center_lon},
)

fig.update_layout(
    title=f"Carte de l'Europe colorée par BLI moyen de {df_bli['year'].min()} à {df_bli['year'].max()}",
    margin={"r":0, "t":50, "l":0, "b":0},
)

fig.show()

<span style="font-size:1.5em;">
    LineCharts
</span>

In [301]:
# Palette de 10 couleurs
colors = px.colors.qualitative.Plotly 

df_top5 = df_bli[df_bli['country'].isin(top5['country'])]
df_bottom5 = df_bli[df_bli['country'].isin(bottom5['country'])]

# Combiner les deux DataFrames pour avoir toutes les lignes
df_combined = pd.concat([df_top5, df_bottom5])
fig = go.Figure()

# Parcourir chaque pays
for i, (country, group) in enumerate(df_combined.groupby('country')):
    group = group.sort_values('year')
    years = group['year'].values
    bli = group['BLI'].values
    country_fr = group['country_fr'].iloc[0]

    mask_valid = ~np.isnan(bli)

    # Tracer la ligne continue pour les valeurs existantes
    fig.add_trace(go.Scatter(
        x=years[mask_valid],
        y=bli[mask_valid],
        mode='lines+markers',
        name=country_fr,
        line=dict(color=colors[i % len(colors)]),  # couleur distincte
        marker=dict(color=colors[i % len(colors)])
    ))

    # Tracer les segments pointillés là où il y a des valeurs manquantes
    for j in range(1, len(years)):
        if mask_valid[j] and not mask_valid[j-1]:
            prev_idx = j - 1
            while prev_idx >= 0 and not mask_valid[prev_idx]:
                prev_idx -= 1
            if prev_idx >= 0:
                fig.add_trace(go.Scatter(
                    x=[years[prev_idx], years[j]],
                    y=[bli[prev_idx], bli[j]],
                    mode='lines',
                    line=dict(dash='dash', color="white"),  # segment pointillé avec même couleur
                    showlegend=False
                ))

# Ajouter la ligne moyenne Europe
df_europe_mean = df_bli.groupby('year')['BLI'].mean().reset_index()
fig.add_trace(go.Scatter(
    x=df_europe_mean['year'],
    y=df_europe_mean['BLI'],
    mode='lines+markers',
    name='Moyenne Europe',
    line=dict(color='black', width=3, dash='dot'),
))

# Mise en page
fig.update_layout(
    title="Évolution du BLI - Les 5 meilleurs et pires",
    xaxis_title="Année",
    yaxis_title="BLI",
    legend_title="Pays",
    template="plotly_white",
    width=900,
    height=500
)

fig.show()

# Mix des deux

In [380]:
df_mix = df_hdi.merge(df_bli, on=['country', 'country_fr', 'cca2', 'cca3', 'year'], how='inner')

In [381]:
df_mix["MIX"] = (df_mix["HDI"] + df_mix["BLI"]) /2
df_mix["MIX"] = (
    (df_mix["MIX"] - df_mix["MIX"].min()) /
    (df_mix["MIX"].max() - df_mix["MIX"].min())
)

In [382]:
nan_par_annee = df_mix.groupby('year')['MIX'].apply(lambda x: x.isna().sum())
annees_valides = nan_par_annee[nan_par_annee < 25].index

df_mix = df_mix[df_mix['year'].isin(annees_valides)].copy()

In [383]:
df_entier = df_mix.copy()

In [384]:
df_mix = df_mix[["country", "country_fr", "cca2", "cca3", "year", "HDI", "BLI", "MIX"]].copy()
print(f"Début des données {df_mix["year"].min()}, et dernière données {df_mix["year"].max()}")

Début des données 2001, et dernière données 2020


In [374]:
# Top 10 des valeurs pour la derniere année 
df_mix_sorted = df_mix[df_mix['year'] == df_mix["year"].max()].sort_values(by="MIX", ascending=False)
print(df_mix_sorted.head(10))

         country country_fr cca2 cca3  year       MIX
704       Sweden      Suède   SE  SWE  2020  0.983353
304      Iceland    Islande   IS  ISL  2020  0.961282
544       Norway    Norvège   NO  NOR  2020  0.946630
720  Switzerland     Suisse   CH  CHE  2020  0.944747
160      Denmark   Danemark   DK  DNK  2020  0.882917
512  Netherlands   Pays-Bas   NL  NLD  2020  0.875055
208      Finland   Finlande   FI  FIN  2020  0.873514
240      Germany  Allemagne   DE  DEU  2020  0.781508
32       Austria   Autriche   AT  AUT  2020  0.766132
320      Ireland    Irlande   IE  IRL  2020  0.762027


In [375]:
# Top 10 des valeurs pour la derniere année 
df_mix_sorted = df_mix[df_mix['year'] == df_mix["year"].max()].sort_values(by="MIX", ascending=True)
print(df_mix_sorted.head(10))

      country country_fr cca2 cca3  year       MIX
0     Albania    Albanie   AL  ALB  2020  0.366460
592   Romania   Roumanie   RO  ROU  2020  0.385994
464   Moldova   Moldavie   MD  MDA  2020  0.386745
560    Poland    Pologne   PL  POL  2020  0.405349
608    Russia     Russie   RU  RUS  2020  0.452235
96   Bulgaria   Bulgarie   BG  BGR  2020  0.478548
112   Croatia    Croatie   HR  HRV  2020  0.482324
656  Slovakia  Slovaquie   SK  SVK  2020  0.534082
352     Italy     Italie   IT  ITA  2020  0.541205
288   Hungary    Hongrie   HU  HUN  2020  0.558649


In [376]:
# Top 5 des valeurs moyennes par pays
df_mix_mean = df_mix.groupby('country')['MIX'].mean().reset_index()

# Top 5 des MIX moyens
top5 = df_mix_mean.nlargest(5, 'MIX')
print("Top 5 des pays par MIX moyen :")
print(top5)

# Bottom 5 des MIX moyens
bottom5 = df_mix_mean.nsmallest(5, 'MIX')
print("\nBottom 5 des pays par MIX moyen :")
print(bottom5)

Top 5 des pays par MIX moyen :
        country       MIX
19      Iceland  0.932851
34       Norway  0.908330
44       Sweden  0.893986
45  Switzerland  0.861289
10      Denmark  0.826144

Bottom 5 des pays par MIX moyen :
                   country       MIX
33         North Macedonia  0.000000
5   Bosnia and Herzegovina  0.139385
40                  Serbia  0.173209
0                  Albania  0.217125
37                 Romania  0.249464


## Affichage

<span style="font-size:1.5em;">
    Cartes
</span>

In [345]:
# Calculer la moyenne MIX
mix_mean = df_mix.groupby('cca3').agg({
    'MIX': 'mean',
    'country_fr': 'first' 
}).reset_index()

# Fusionner les données MIX avec le GeoDataFrame
europe_mix = europe.merge(mix_mean, left_on='id', right_on='cca3', how='left')

# Liste des pays avec MIX
with_mix = europe_mix[~europe_mix['MIX'].isna()]
without_mix = europe_mix[europe_mix['MIX'].isna()]

# Centre de la carte
center_lon = (45 + -25) / 2
center_lat = (72 + 34) / 2

# Création de la figure Plotly
fig = go.Figure()

# Couche MIX
fig.add_trace(go.Choropleth(
    geojson=with_mix.__geo_interface__,
    locations=with_mix.index,
    z=with_mix['MIX'],
    text=with_mix['country_fr'],
    colorscale='YlGn',
    colorbar_title="MIX moyen",
    marker_line_color='white',
    marker_line_width=0.5,
    hovertemplate="<b>%{text}</b><br>MIX: %{z}<extra></extra>",  # Info bulle pour MIX connu
))

# Couche gris pour pays sans MIX
fig.add_trace(go.Choropleth(
    geojson=without_mix.__geo_interface__,
    locations=without_mix.index,
    z=[0]*len(without_mix),  # valeur arbitraire
    text=without_mix['country_fr'],
    colorscale=[[0, 'lightgrey'], [1, 'lightgrey']],  # gris fixe
    showscale=False,
    marker_line_color='white',
    marker_line_width=0.5,
    hovertemplate="<b>%{text}</b><br>MIX: Null<extra></extra>",  # Info bulle pour MIX manquant
))

fig.update_geos(
    visible=False,
    projection_scale=5,
    center={"lat": center_lat, "lon": center_lon},
)

fig.update_layout(
    title=f"Carte de l'Europe colorée par MIX moyen de {df_mix['year'].min()} à {df_mix['year'].max()}",
    margin={"r":0, "t":50, "l":0, "b":0},
)

fig.show()

<span style="font-size:1.5em;">
    LineCharts
</span>

In [440]:
# Palette de 10 couleurs
colors = px.colors.qualitative.Plotly  

df_top5 = df_mix[df_mix['country'].isin(top5['country'])]
df_bottom5 = df_mix[df_mix['country'].isin(bottom5['country'])]

# Combiner les deux DataFrames pour avoir toutes les lignes
df_combined = pd.concat([df_top5, df_bottom5])
fig = go.Figure()

# Parcourir chaque pays
for i, (country, group) in enumerate(df_combined.groupby('country')):
    group = group.sort_values('year')
    years = group['year'].values
    mix = group['MIX'].values
    country_fr = group['country_fr'].iloc[0]

    mask_valid = ~np.isnan(mix)

    # Tracer la ligne continue pour les valeurs existantes
    fig.add_trace(go.Scatter(
        x=years[mask_valid],
        y=mix[mask_valid],
        mode='lines+markers',
        name=country_fr,
        line=dict(color=colors[i % len(colors)]),  # couleur distincte
        marker=dict(color=colors[i % len(colors)])
    ))

    # Tracer les segments pointillés là où il y a des valeurs manquantes
    for j in range(1, len(years)):
        if mask_valid[j] and not mask_valid[j-1]:
            prev_idx = j - 1
            while prev_idx >= 0 and not mask_valid[prev_idx]:
                prev_idx -= 1
            if prev_idx >= 0:
                fig.add_trace(go.Scatter(
                    x=[years[prev_idx], years[j]],
                    y=[mix[prev_idx], mix[j]],
                    mode='lines',
                    line=dict(dash='dash', color="white"),  # segment pointillé avec même couleur
                    showlegend=False
                ))

# Ajouter la ligne moyenne Europe
df_europe_mean = df_mix.groupby('year')['MIX'].mean().reset_index()
fig.add_trace(go.Scatter(
    x=df_europe_mean['year'],
    y=df_europe_mean['MIX'],
    mode='lines+markers',
    name='Moyenne Europe',
    line=dict(color='black', width=3, dash='dot'),
))

# Mise en page
fig.update_layout(
    title="Évolution du MIX - Les 5 meilleurs et pires",
    xaxis_title="Année",
    yaxis_title="MIX",
    legend_title="Pays",
    template="plotly_white",
    width=900,
    height=500
)

fig.show()

In [452]:
# Moyenne par pays
df_mean = df_mix.groupby('country').agg({
    'HDI': 'mean',
    'BLI': 'mean',
    'MIX': 'mean',
    'country_fr': 'first'
}).reset_index()

fig = px.scatter(
    df_mean,
    x="HDI",
    y="BLI",
    custom_data=["country_fr", "HDI", "BLI", "MIX"],  # 👈 IMPORTANT
    title="Relation entre HDI et BLI (moyenne par pays)"
)

# Style + hover personnalisé
fig.update_traces(
    marker=dict(color="royalblue", size=10),
    hovertemplate=
    "<b>%{customdata[0]}</b><br>" +
    "HDI: %{customdata[1]:.3f}<br>" +
    "BLI: %{customdata[2]:.3f}<br>" +
    "MIX: %{customdata[3]:.3f}<extra></extra>"
)

fig.update_layout(
    showlegend=False,
    xaxis_title="HDI (moyenne)",
    yaxis_title="BLI (moyenne)",
    template="plotly_white",
    width=900,
    height=500
)

fig.show()

In [460]:
def plot_mix_pays(df_mix, pays_list, col_filter="cca2"):
    colors = px.colors.qualitative.Plotly  

    # Filtrage dynamique
    df_filtered = df_mix[df_mix[col_filter].isin(pays_list)]

    if df_filtered.empty:
        print("Aucune donnée pour les pays fournis.")
        return

    fig = go.Figure()

    # Parcourir chaque pays
    for i, (country, group) in enumerate(df_filtered.groupby(col_filter)):
        group = group.sort_values('year')
        years = group['year'].values
        mix = group['MIX'].values
        country_fr = group['country_fr'].iloc[0]

        mask_valid = ~np.isnan(mix)

        color = colors[i % len(colors)]

        # Ligne principale
        fig.add_trace(go.Scatter(
            x=years[mask_valid],
            y=mix[mask_valid],
            mode='lines+markers',
            name=country_fr,
            line=dict(color=color),
            marker=dict(color=color)
        ))

        # Segments pointillés (trous)
        for j in range(1, len(years)):
            if mask_valid[j] and not mask_valid[j-1]:
                prev_idx = j - 1
                while prev_idx >= 0 and not mask_valid[prev_idx]:
                    prev_idx -= 1
                if prev_idx >= 0:
                    fig.add_trace(go.Scatter(
                        x=[years[prev_idx], years[j]],
                        y=[mix[prev_idx], mix[j]],
                        mode='lines',
                        line=dict(dash='dash', color=color),  # 👈 même couleur (corrigé)
                        showlegend=False
                    ))

    # Moyenne Europe (optionnelle)
    year_min = df_filtered['year'].min()
    year_max = df_filtered['year'].max()
    df_europe_mean = (
        df_mix[df_mix['year'].between(year_min, year_max)]
        .groupby('year')['MIX']
        .mean()
        .reset_index()
    )
    fig.add_trace(go.Scatter(
        x=df_europe_mean['year'],
        y=df_europe_mean['MIX'],
        mode='lines+markers',
        name='Moyenne Europe',
        line=dict(color='black', width=3, dash='dot'),
    ))

    fig.update_layout(
        title="Évolution du MIX - Pays sélectionnés",
        xaxis_title="Année",
        yaxis_title="MIX",
        legend_title="Pays",
        template="plotly_white",
        width=900,
        height=500
    )

    fig.show()

In [ ]:
plot_mix_pays(df_mix, ["FR", "GB"]) 

# Conclusion

In [402]:
def get_best_indicators(df_mix, year):

    df = df_mix.copy()
    df.columns = df.columns.str.strip()

    # filtrer année + données valides
    df = df[df["year"] == year]
    df = df.dropna(subset=["HDI", "BLI", "MIX"])

    if df.empty:
        return f"Aucune donnée pour l'année {year}"

    result = {}

    for col in ["HDI", "BLI", "MIX"]:

        # rangs dans cette année uniquement
        df[f"rank_{col}"] = df[col].rank(ascending=False)

        # meilleur pays pour cet indicateur
        best_row = df.loc[df[col].idxmax()]

        result[col] = {
            "country": best_row["country_fr"],
            "value": best_row[col],
            "rank": int(best_row[f"rank_{col}"])
        }

    return pd.DataFrame(result).T

In [403]:
get_best_indicators(df_mix, 2020)

,country,value,rank
HDI,Suède,0.929605,1
BLI,Suède,5.381782,1
MIX,Suède,0.983353,1


In [404]:
def get_country_ranks(df_mix, country_name, year):

    df_mix = df_mix.copy()
    df_mix.columns = df_mix.columns.str.strip()

    # garder uniquement les données valides + année demandée
    df_mix = df_mix.dropna(subset=["HDI", "BLI", "MIX"])
    df_mix = df_mix[df_mix["year"] == year]

    # calcul des rangs (1 = meilleur) sur CETTE année uniquement
    df_mix["rank_HDI"] = df_mix["HDI"].rank(ascending=False)
    df_mix["rank_BLI"] = df_mix["BLI"].rank(ascending=False)
    df_mix["rank_MIX"] = df_mix["MIX"].rank(ascending=False)

    # filtrer le pays
    res = df_mix[df_mix["country_fr"].str.lower() == country_name.lower()]

    if res.empty:
        return f"Pays '{country_name}' introuvable pour l'année {year}"

    row = res.iloc[0]

    return pd.DataFrame([{
        "country": row["country_fr"],
        "year": int(row["year"]),
        "HDI": row["HDI"],
        "rang_HDI": int(row["rank_HDI"]),
        "BLI": row["BLI"],
        "rang_BLI": int(row["rank_BLI"]),
        "MIX": row["MIX"],
        "rang_MIX": int(row["rank_MIX"]),
    }])

In [405]:
get_country_ranks(df_mix, "France", 2020)

,country,year,HDI,rang_HDI,BLI,rang_BLI,MIX,rang_MIX
0,France,2020,0.849715,11,4.62816,15,0.691174,15


In [406]:
def get_country_best_indicators(df_mix, country_name):

    df = df_mix.copy()
    df.columns = df.columns.str.strip()

    # garder le pays
    df = df[df["country_fr"].str.lower() == country_name.lower()]

    if df.empty:
        return f"Pays '{country_name}' introuvable"

    # supprimer NaN
    df = df.dropna(subset=["HDI", "BLI", "MIX"])

    result = {}

    for col in ["HDI", "BLI", "MIX"]:

        # meilleure année pour CE pays
        best_row = df.loc[df[col].idxmax()]

        # rang du pays cette année-là (comparé aux autres pays)
        df_temp = df_mix.copy()
        df_temp.columns = df_temp.columns.str.strip()
        df_temp = df_temp.dropna(subset=["HDI", "BLI", "MIX"])
        df_temp = df_temp[df_temp["year"] == best_row["year"]]

        df_temp[f"rank_{col}"] = df_temp[col].rank(ascending=False)

        rank = df_temp[df_temp["country"].str.lower() == country_name.lower()][f"rank_{col}"].values[0]

        result[col] = {
            "year": int(best_row["year"]),
            "value": best_row[col],
            "rank": int(rank)
        }

    return pd.DataFrame(result).T

In [407]:
get_country_best_indicators(df_mix, "France")

,year,value,rank
HDI,2019.0,0.854828,11.0
BLI,2020.0,4.628160,15.0
MIX,2019.0,0.692429,15.0
